# Notebook 086: Unity Catalog Three-Tier Namespace: Catalogs, Schemas and Managed vs External Tables

**Module:** Module 09: Databricks Platform, Unity Catalog, Security & FinOps  
**Difficulty:** `Intermediate`  
**Databricks Masterclass Reference:** Video 03: Databricks Architecture & Video 14: Master Unity Catalog in Databricks  

---

## 1. Business Scenario & Real-World Context
An enterprise migrates from the legacy Hive Metastore (`hive_metastore.default`) to Databricks Unity Catalog. Platform administrators need to establish a secure 3-tier namespace (`catalog.schema.table`) separating dev, staging, and prod environments, and differentiate between Managed and External tables.

## 2. Technical Requirement & Problem Statement
Master Unity Catalog architecture: create 3-tier namespaces (`catalog.schema.table`), configure Managed tables (storage lifecycle managed by Databricks) vs External tables (pointing to existing cloud object storage paths), grant access permissions, and query table metadata.

## 3. Spark & Databricks Architecture Notes
Unity Catalog introduces a centralized governance model across workspaces using a 3-tier namespace: `catalog` -> `schema` -> `table/view/volume`. Dropping a Managed table deletes both metadata and underlying Delta files; dropping an External table removes ONLY metadata, preserving underlying cloud storage data files.


## 4. Environment Setup & SparkSession Initialization
We configure a resilient `SparkSession` compatible with local PySpark development, Databricks runtime, or CI/CD test runners.

In [ ]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

# Initialize SparkSession with Delta Lake support and optimized local configs
spark = (
    SparkSession.builder
    .appName(f"UseCase_086_Unity_Catalog_Three-Tier_Namespace_Catalogs,_Schemas_and_Managed_vs_External_Tables")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.default.parallelism", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f" Spark Version: {spark.version}")
print(f" Session Active: {spark.sparkContext.appName}")


## 5. Configuring 3-Tier Namespace Hierarchy in SQL
Create catalogs and schemas representing development and production environments.

In [ ]:
# SQL Blueprint for Unity Catalog 3-Tier Hierarchy:
uc_setup_sql = """
-- 1. Create Catalogs for environment isolation
CREATE CATALOG IF NOT EXISTS enterprise_dev COMMENT 'Development Sandbox Catalog';
CREATE CATALOG IF NOT EXISTS enterprise_prod COMMENT 'Production Analytical Catalog';

-- 2. Create Schemas within Catalog
CREATE SCHEMA IF NOT EXISTS enterprise_dev.finance COMMENT 'Finance development schema';
CREATE SCHEMA IF NOT EXISTS enterprise_dev.marketing COMMENT 'Marketing development schema';

-- 3. Set current active context
USE CATALOG enterprise_dev;
USE SCHEMA finance;
"""

print("Unity Catalog 3-Tier DDL Hierarchy defined:")
print(uc_setup_sql)

## 6. Comparing Managed vs External Tables
Illustrate differences in storage lifecycle and DDL syntax.

In [ ]:
# Managed Table: Storage location managed by Unity Catalog metastore root
# CREATE TABLE enterprise_dev.finance.t_managed_invoices (
#     invoice_id STRING,
#     amount DOUBLE
# ) USING DELTA;

# External Table: References explicit cloud storage location (S3 / ADLS / GCS)
# CREATE TABLE enterprise_dev.finance.t_external_invoices (
#     invoice_id STRING,
#     amount DOUBLE
# ) USING DELTA
# LOCATION 's3://my-enterprise-data-bucket/finance/invoices/';

table_comparison = [
    ("Managed Table", "Managed by Unity Catalog root", "Both metadata AND physical data files are DELETED"),
    ("External Table", "Custom user-supplied S3/ADLS/GCS URL", "Metadata deleted; physical Parquet files are PRESERVED")
]

schema = "table_type STRING, storage_lifecycle STRING, drop_table_behavior STRING"
df_comp = spark.createDataFrame(table_comparison, schema)
df_comp.show(truncate=False)

## 7. Granting Access Control Privileges
Apply role-based access control grants in Unity Catalog.

In [ ]:
# Production Grant Examples:
# GRANT USAGE ON CATALOG enterprise_dev TO `data_engineers`;
# GRANT SELECT, MODIFY ON SCHEMA enterprise_dev.finance TO `data_engineers`;
# GRANT SELECT ON TABLE enterprise_dev.finance.t_managed_invoices TO `finance_analysts`;

print("Unity Catalog RBAC privilege model mapped.")

## 9. Key Architectural Takeaways
- **Unity Catalog replaces the legacy 2-tier Hive Metastore with a structured 3-tier namespace (`catalog.schema.table`).**
- **Managed tables manage both file storage and metadata; dropping them purges physical data files.**
- **External tables decouple metadata from underlying cloud storage, preserving files upon DROP.**

## 10. Senior Data Engineering Interview Questions
1. What is the difference between Managed Tables and External Tables in Databricks Unity Catalog?
2. What happens to underlying data files when an External Table is dropped versus a Managed Table?
3. Why is a 3-tier namespace superior to the legacy 2-tier Hive Metastore for enterprise multi-tenancy?

---
*Created for the **Spark Coding Use Cases Masterclass** repository referencing the Databricks All-in-One Playlist.*
